In [ ]:
# Import functions from preprocessing.py
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.optim as optim
from tqdm import tqdm

from GPModel import GPModel
from GPArealModel import GPArealModel
from VIGP_Unlinked import VIGP_Unlinked


# Add the path to the src directory
sys.path.append(os.path.abspath(os.path.join('..', 'data')))

result = {}
B = 100
n_i = 4
seed = 1
input_dim = 1
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

niter_GP=2000
niter_GPAreal=2000
niter_VI= 2000

# Load data from the specified path
data_path = os.path.join('..', 'data', f'B_{B}_n_{n_i}', f'data_seed_{seed}.pt')
data = torch.load(data_path)

# Extract variables from the data dictionary
y = data['y']
region_assignments = data['region_assignments']
x = data['x']
w = data['w']
e = data['e']
s = data['s']
x_jumbled_within_regions = data['x_jumbled_within_regions']
s_jumbled_within_regions = data['s_jumbled_within_regions']
perm_matrix_x = data['perm_matrix_x']
perm_matrix_s = data['perm_matrix_s']
sigmasq_true = data['sigmasq_true']
phi_true = data['phi_true']
beta_true = data['beta_true']
nu_true = data['nu_true']
tausq_true = data['tausq_true']

# Train the GPmodel (oracle)
# Oracle GP model if the locations and links are known
# Initialize and optimize the model
model = GPModel().to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)

for i in tqdm(range(niter_GP)):
    optimizer.zero_grad()
    loss = model(s, x, y)
    loss.backward()
    optimizer.step()
    
    # Constrain sigmasq, length_scale, and tausq to be positive
    with torch.no_grad():
        model.sigmasq.clamp_(min=1e-6)
        model.length_scale.clamp_(min=1e-6)
        model.tausq.clamp_(min=1e-6)
        
# Save the model parameters
model_params = {
    'nu': model.nu.item(),
    'phi': 1/model.length_scale.item(),
    'sigmasq': model.sigmasq.item(),
    'tausq': model.tausq.item(),
    'beta': model.beta.detach().cpu().numpy()
}

# Save the model parameters to a file
result['GPmodel'] = model_params



# Train the model GPareal
# Compute region-wise averages directly
unique_regions = torch.unique(region_assignments)
B = len(unique_regions)

ybar = torch.zeros(B, device=y.device)
xbar = torch.zeros(B, input_dim, device=x.device)

for i, region in enumerate(unique_regions):
    # Get indices for the current region
    indices = torch.where(region_assignments == region)[0]
    
    # Compute region-wise averages for y and x
    ybar[i] = torch.mean(y[indices])
    xbar[i] = torch.mean(x_jumbled_within_regions[indices], dim=0)


# Initialize and optimize the model
model = GPArealModel().to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)

for i in tqdm(range(niter_GPAreal)):
    optimizer.zero_grad()
    loss = model(s_jumbled_within_regions, region_assignments, xbar, ybar)
    loss.backward()
    optimizer.step()


# Save the model parameters
model_params = {
    'nu': model.nu.item(),
    'phi': 1/model.length_scale.item(),
    'sigmasq': model.sigmasq.item(),
    'tausq': model.tausq.item(),
    'beta': model.beta.detach().cpu().numpy()
}
# Save the model parameters to a file
result['GPArealModel'] = model_params


# Train the model VIGP_unlinked
niter_VI = 10
n_blocks = B
n_locations = n_i

# Random toy data for X and Y
X = torch.tensor(x_jumbled_within_regions, dtype=torch.float32).reshape(n_blocks, n_locations)
Y = torch.tensor(y, dtype=torch.float32).reshape(n_blocks, n_locations)

# Generate (n_blocks * n_locations) 2D coordinates
total_points = n_blocks * n_locations
locations = torch.tensor(s_jumbled_within_regions,dtype=torch.float32)

# Compute distance matrix from locations
Dist = torch.cdist(locations, locations, p=2)  # Pairwise distances

# Set optional args
n_steps = 50
n_phi_samples = 50
n_piX_sample = 50
tau_X = 0.3
tau_S = 0.3
n_piS_sample = 50

for tau in [0.1, 0.3, 0.5]:
    tau_X = tau
    tau_S = tau

    results_VI = VIGP_Unlinked(
        n_iter=niter_VI,
        n_blocks=n_blocks,
        n_locations=n_locations,
        phi_prior_ub=5,
        X=X,
        Y=Y,
        Dist=Dist,
        n_steps=n_steps,
        n_phi_samples=n_phi_samples,
        n_piX_sample=n_piX_sample,
        tau_X=tau_X, tau_S=tau_S,
        n_piS_sample=n_piS_sample,
        seed=521, 
        fix_piX= False, 
        fix_piS= False,
        fix_mu_lambda_beta=False,
        fix_sigmasq_lambda_beta=False,
        fix_lambda_b1=False,
        lambda_b1_fixed=((B*n_i)*0.5 + 0.1) * 5,
        fix_lambda_b2=False,
        M_X_star_fixed=perm_matrix_x.T,
        M_S_star_fixed=perm_matrix_s.T,
        V_X_star_fixed=torch.eye(n_locations, n_locations, device=device),
        V_S_star_fixed=torch.eye(n_locations, n_locations, device=device), 
        phi_init = 0.5,
        mean_Rphi_inv_fixed= torch.linalg.inv(torch.exp(-4 * Dist)),
        fix_mean_Rphi_inv=False, 
        pi_X_true = perm_matrix_x.T,
        pi_S_true = perm_matrix_s.T,
        VX_ub = 0.5,
        VS_ub=0.5
    )
   
    # Save the model parameters to the result dictionary
    result[f'VIGP_unlinked_tau_{tau}'] = results_VI
   
# Save the result dictionary to a file
result_path = os.path.join('..', 'data', 'results' , f'B_{B}_n_{n_i}', f'results_seed_{seed}.pt')
os.makedirs(os.path.dirname(result_path), exist_ok=True)
torch.save(result, result_path)

/var/folders/w7/jxz2zn316391355qstwl03940000gn/T/ipykernel_6532/1681579770.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(data_path)
  0%|          |

Minimum eigenvalue of V_X_star - M_X_star.T @ M_X_star: 1.1921e-07
Minimum eigenvalue of V_S_star - M_S_star.T @ M_S_star: 1.1921e-07


 10%|█         | 1/10 [00:23<03:30, 23.41s/it]

Iter 1/10 | mu_lambda_beta: 5.7557 | 
 sigmasq_lambda_beta: 0.0000 | 
 lambda_a1: 200.1000 | lambda_b1: 16841.6191 | lambda_a2: 200.1000 | lambda_b2: 2213.3074
‣  E[ϕ]: 0.8185 | ‣ ||mu_W||: 28.7475
Number of correct permutations recognized for piX: 4.0
Number of correct permutations recognized for piS: 4.0
Total Loss: 2.2207
Minimum eigenvalue of V_X_star - M_X_star.T @ M_X_star: 1.8791e-02
Minimum eigenvalue of V_S_star - M_S_star.T @ M_S_star: 1.8556e-02


 20%|██        | 2/10 [00:47<03:10, 23.77s/it]

...

Iter 8/10 | mu_lambda_beta: 7.2605 | 
 sigmasq_lambda_beta: 0.0139 | 
 lambda_a1: 200.1000 | lambda_b1: 2618.4443 | lambda_a2: 200.1000 | lambda_b2: 330.2251
‣  E[ϕ]: 3.1879 | ‣ ||mu_W||: 32.4378
Number of correct permutations recognized for piX: 4.0
Number of correct permutations recognized for piS: 4.0
Total Loss: 1.2498
Minimum eigenvalue of V_X_star - M_X_star.T @ M_X_star: 8.0415e-03
Minimum eigenvalue of V_S_star - M_S_star.T @ M_S_star: 1.4585e-02


 90%|█████████ | 9/10 [03:48<00:25, 25.50s/it]

Iter 9/10 | mu_lambda_beta: 7.3161 | 
 sigmasq_lambda_beta: 0.0124 | 
 lambda_a1: 200.1000 | lambda_b1: 2467.3657 | lambda_a2: 200.1000 | lambda_b2: 312.1577
‣  E[ϕ]: 3.0463 | ‣ ||mu_W||: 31.9882
Number of correct permutations recognized for piX: 4.0
Number of correct permutations recognized for piS: 4.0
Total Loss: 1.2135
Minimum eigenvalue of V_X_star - M_X_star.T @ M_X_star: 8.0205e-03
Minimum eigenvalue of V_S_star - M_S_star.T @ M_S_star: 1.4888e-02


100%|██████████| 10/10 [04:14<00:00, 25.43s/it]

Iter 10/10 | mu_lambda_beta: 7.3648 | 
 sigmasq_lambda_beta: 0.0117 | 
 lambda_a1: 200.1000 | lambda_b1: 1988.7146 | lambda_a2: 200.1000 | lambda_b2: 294.4243
‣  E[ϕ]: 3.5934 | ‣ ||mu_W||: 31.7439
Number of correct permutations recognized for piX: 4.0
Number of correct permutations recognized for piS: 4.0
Total Loss: 1.1828


In [2]:
results_VI['loss_vector']

tensor([4.0859, 3.0166, 2.8421, 2.4574, 2.1250, 1.8667, 1.6540, 1.5619, 1.4727,
        1.3989])